# 02 - Elephant Threshold Selection

This notebook derives the μ + 3·σ (Chebyshev) elephant cutoff from the CIC-IDS2017 working set and contrasts it with percentile-based thresholds.

**Outputs**
- Figure 4: empirical CDF of bidirectional bytes with the Chebyshev cutoff annotated.
- Figure 5: linear and log histograms of flow sizes.
- Table 2: 90th/95th/98th/99th/99.5th percentiles vs the Chebyshev bytes threshold.
- Figure 6: mice vs elephant ratio under μ + 3·σ.
- Markdown discussion contrasting the observed ≈0.09% elephants with the ≈5% from the proprietary trace.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.style.use("seaborn-v0_8-colorblind")

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "data" / "intermediate" / "cicids2017_paper_base_glf.csv"
THRESHOLD_PATH = PROJECT_ROOT / "artifacts" / "threshold.json"
for p in (DATA_PATH, THRESHOLD_PATH):
    assert p.exists(), f"Missing required file: {p}"

data = pd.read_csv(DATA_PATH)
data.columns = data.columns.str.strip()
bytes_series = pd.to_numeric(data["bidirectional_bytes"], errors="coerce").fillna(0.0)
with open(THRESHOLD_PATH) as fh:
    threshold_info = json.load(fh)
chebyshev_threshold = threshold_info["threshold_bytes"]
chebyshev_fraction = threshold_info["elephant_fraction"]
DATA_PATH, THRESHOLD_PATH

In [ ]:
sorted_bytes = np.sort(bytes_series.values)
cdf = np.linspace(0, 1, len(sorted_bytes), endpoint=False)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(sorted_bytes, cdf, color="#4c72b0", label="Empirical CDF")
ax.axvline(chebyshev_threshold, color="#dd8452", linestyle="--", label="μ + 3σ")
ax.set_xlabel("Bidirectional bytes")
ax.set_ylabel("Cumulative share of flows")
ax.set_title("Figure 4 – Flow-size CDF with Chebyshev cutoff")
ax.legend()
ax.grid(True, ls="--", alpha=0.3)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
axes[0].hist(bytes_series, bins=200, color="#4c72b0", alpha=0.85)
axes[0].set_title("Figure 5a – Histogram (linear scale)")
axes[0].set_xlabel("Bidirectional bytes")
axes[0].set_ylabel("Flow count")
axes[0].grid(True, ls="--", alpha=0.3)

axes[1].hist(bytes_series.clip(lower=1), bins=200, color="#55a868", alpha=0.85)
axes[1].set_xscale("log")
axes[1].set_title("Figure 5b – Histogram (log scale)")
axes[1].set_xlabel("Bidirectional bytes (log)")
axes[1].grid(True, which="both", ls="--", alpha=0.3)

for ax in axes:
    ax.axvline(chebyshev_threshold, color="#dd8452", linestyle="--", label="μ + 3σ")
axes[1].legend(loc="upper right")
plt.tight_layout()
plt.show()

In [ ]:
quantile_levels = [0.90, 0.95, 0.98, 0.99, 0.995]
quantiles = bytes_series.quantile(quantile_levels)
summary_rows = []
for lvl, q in quantiles.items():
    summary_rows.append(
        {
            "Threshold": f"{int(lvl * 100)}th percentile",
            "Bytes": f"{q:,.0f}",
            "Share of flows above": f"{(1 - lvl) * 100:.3f}%",
        }
    )
summary_rows.append(
    {
        "Threshold": "μ + 3σ (Chebyshev)",
        "Bytes": f"{chebyshev_threshold:,.0f}",
        "Share of flows above": f"{chebyshev_fraction * 100:.3f}%",
    }
)
table2 = pd.DataFrame(summary_rows)
table2

In [ ]:
elephants = int(len(bytes_series) * chebyshev_fraction)
mice = len(bytes_series) - elephants
labels = ["Mice", "Elephants"]
values = [mice, elephants]
percentages = [value / len(bytes_series) * 100 for value in values]

fig, ax = plt.subplots(figsize=(5, 4))
positions = range(len(labels))
ax.bar(positions, percentages, color=["#4c72b0", "#dd8452"])
ax.set_xticks(positions)
ax.set_xticklabels(labels)
ax.set_ylabel("Share of flows (%)")
ax.set_ylim(0, max(percentages) * 1.2)
ax.set_title("Figure 6 – Flow ratio under μ + 3σ")
for idx, (pct, val) in enumerate(zip(percentages, values)):
    ax.text(idx, pct + 0.02, f"{pct:.3f}% ({val})", ha="center", va="bottom")
plt.tight_layout()
plt.show()

Only ≈0.09% of CIC-IDS2017 flows exceed μ + 3σ (49 / 55,726), which is dramatically lower than the ≈5.11% elephant share cited in the proprietary trace from the paper. This highlights how traffic mix drives the cutoff: the public dataset is dominated by mice, so percentile-based thresholds would classify far more flows as elephants than the Chebyshev rule used by the authors.